# MammoDiffusion LDM - 04b2 SD-VAE extra1361

Nuovo esperimento from-scratch: la U-Net LDM viene addestrata da zero sui latenti prodotti dal VAE preaddestrato di Stable Diffusion 2.1.

Rispetto a `06_LDM_Extra1361_FromScratch.ipynb`:
- non viene addestrato un VAE custom;
- i latenti train/validation sono prodotti da `AutoencoderKL`;
- la U-Net viene comunque riaddestrata completamente;
- le immagini filtrate finali vengono salvate in `data/synthetic/fromscratch_new/{positive, negative}`.


## 1. Selezione GPU


In [1]:
# === Bootstrap unificato notebooks/ ===
# Funziona dalla root del progetto e da ogni sottocartella della struttura notebooks/.
import sys as _sys
from pathlib import Path as _Path


def _find_mammo_root():
    for _candidate in [_Path.cwd().resolve(), *_Path.cwd().resolve().parents]:
        if _candidate.name == "MammoDiffusion":
            return _candidate
        if (_candidate / "data").is_dir() and (_candidate / "notebooks").is_dir():
            return _candidate
    raise FileNotFoundError("Root MammoDiffusion non trovata da " + str(_Path.cwd()))


PROJECT_ROOT = _find_mammo_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
UTILITY_DIR = NOTEBOOKS_DIR / "utility"
for _path in (str(UTILITY_DIR), str(NOTEBOOKS_DIR)):
    if _path not in _sys.path:
        _sys.path.insert(0, _path)

BASE = PROJECT_ROOT
BASE_DIR = PROJECT_ROOT
BASE_PATH = str(PROJECT_ROOT) + "/"
# === Fine bootstrap unificato ===

import os
import subprocess
import sys
from pathlib import Path

CUDA_ROOT = Path(os.environ.get("MAMMODIFFUSION_CUDA_ROOT", os.environ.get("CONDA_PREFIX", sys.prefix)))

libdevice_path = CUDA_ROOT / "nvvm" / "libdevice" / "libdevice.10.bc"
if libdevice_path.exists():
    os.environ["XLA_FLAGS"] = f"--xla_gpu_cuda_data_dir={CUDA_ROOT}"

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
        check=False,
        capture_output=True,
        text=True,
    )
    if result.stdout.strip():
        print("GPU fisiche disponibili:")
        print(result.stdout.strip())
except FileNotFoundError:
    print("nvidia-smi non disponibile.")

print("XLA_FLAGS:", os.environ.get("XLA_FLAGS", ""))

# GPU dedicata al training; la generazione usa invece GENERATION_GPU_DEVICES nei subprocess.
TRAIN_GPU_VISIBLE_DEVICES = "0"

def training_subprocess_env():
    env = os.environ.copy()
    if TRAIN_GPU_VISIBLE_DEVICES is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(TRAIN_GPU_VISIBLE_DEVICES)
    return env

# Generazione multi-GPU (non influenza il training).
PARALLEL_GENERATION = True
GENERATION_GPU_DEVICES = "auto"
GENERATION_MAX_WORKERS = None

def add_generation_parallel_args(command):
    if PARALLEL_GENERATION:
        command.extend(["--generation-gpus", GENERATION_GPU_DEVICES])
        if GENERATION_MAX_WORKERS is not None:
            command.extend(["--max-generation-workers", str(GENERATION_MAX_WORKERS)])
    else:
        command.extend(["--generation-gpus", "off"])
    return command

print("CUDA_VISIBLE_DEVICES ereditato:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("GENERATION_GPU_DEVICES richiesto:", GENERATION_GPU_DEVICES)
from parallel_generation_utils import print_gpu_resolution_dry_run
print_gpu_resolution_dry_run(GENERATION_GPU_DEVICES, GENERATION_MAX_WORKERS)


GPU fisiche disponibili:
0, NVIDIA GeForce RTX 3060, 12288 MiB
1, NVIDIA GeForce RTX 5060 Ti, 16311 MiB
XLA_FLAGS: --xla_gpu_cuda_data_dir=/home/fede/miniforge3/envs/tf-gpu
CUDA_VISIBLE_DEVICES ereditato: None
GENERATION_GPU_DEVICES richiesto: auto
GPU fisiche (nvidia-smi): ['0', '1']
CUDA_VISIBLE_DEVICES ereditato: None
GPU richieste (--generation-gpus): auto
GPU risolte: ['0', '1']
Numero worker: 2


['0', '1']

## 2. Setup dipendenze


In [2]:
!pip install -q pandas numpy matplotlib scikit-learn pillow gdown tensorflow scikit-image scipy psutil codecarbon torch torchvision torchmetrics torch-fidelity prdc diffusers transformers accelerate safetensors

## 3. Path progetto ed esperimento


In [3]:
# Setup must precede the phase planner; this cell only defines paths/helpers.
if True:
    from pathlib import Path
    from tempfile import TemporaryDirectory
    import os
    import shutil
    import subprocess
    import sys
    import time
    import zipfile

    import gdown

    PROJECT_NAME = "MammoDiffusion"
    EXPERIMENT_NAME = "diffusers/07_ldm_sdvae_extra1361"
    RESULTS_STAGE_NAME = "diffusers/07_ldm_sdvae_extra1361"

    SD21_MODEL_DRIVE_ID = "10XRn-bxpp7tP6ROWLYpeCNYJZIaHfMUt"
    FORCE_MODEL_REDOWNLOAD = False
    PROJECT_ROOT_OVERRIDE = None


    def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
        if override is not None:
            root = Path(override).expanduser().resolve()
            if not root.exists():
                raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
            return root

        cwd = Path.cwd().resolve()
        for candidate in [cwd, *cwd.parents]:
            if candidate.name == project_name:
                return candidate
            has_notebooks = (candidate / "notebooks").exists() or (candidate / "notebooks").exists()
            if ((candidate / "data").exists() and has_notebooks) or ((candidate / ".git").exists() and has_notebooks):
                return candidate
        for candidate in [
            cwd / project_name,
            Path("/content") / project_name,
            Path("/content/drive/MyDrive") / project_name,
            Path.home() / project_name,
        ]:
            if candidate.exists():
                return candidate.resolve()
        raise FileNotFoundError("Root MammoDiffusion non trovata.")


    PROJECT_ROOT = find_project_root()
    NOTEBOOKS_DIR = PROJECT_ROOT / ("notebooks" if (PROJECT_ROOT / "notebooks").is_dir() else "notebooks")
    UTILITY_DIR = NOTEBOOKS_DIR / "utility" if (NOTEBOOKS_DIR / "utility").is_dir() else NOTEBOOKS_DIR
    DATA_DIR = PROJECT_ROOT / "data"
    DATA_PROCESSED_DIR = DATA_DIR / "processed"
    EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / EXPERIMENT_NAME
    SHARED_PRETRAINED_ROOT = NOTEBOOKS_DIR / "pretrained_model"
    PRETRAINED_MODEL_DIR = SHARED_PRETRAINED_ROOT / "stable-diffusion-2-1-base"
    PRETRAINED_MODEL_ZIP_PATH = SHARED_PRETRAINED_ROOT / "archives" / "stable-diffusion-2-1-base.zip"
    SD_VAE_MODEL_NAME_OR_PATH = PRETRAINED_MODEL_DIR
    MODELS_DIR = EXPERIMENT_DIR / "models"
    CHECKPOINTS_DIR = EXPERIMENT_DIR / "checkpoints_ldm"
    LATENTS_DIR = EXPERIMENT_DIR / "latents"
    LOGS_DIR = EXPERIMENT_DIR / "logs"
    RESULTS_DIR = PROJECT_ROOT / "results" / RESULTS_STAGE_NAME
    RESULTS_PLOTS_DIR = RESULTS_DIR / "plots"
    RESULTS_METRICS_DIR = RESULTS_DIR / "metrics"
    RESULTS_ECOTRACKER_DIR = RESULTS_DIR / "ecotracker"
    SYNTHETIC_NEW_DIR = DATA_DIR / "synthetic" / "fromscratch_new"
    SYNTHETIC_NEW_POS_DIR = SYNTHETIC_NEW_DIR / "positive"
    SYNTHETIC_NEW_NEG_DIR = SYNTHETIC_NEW_DIR / "negative"

    for directory in [
        EXPERIMENT_DIR, PRETRAINED_MODEL_ZIP_PATH.parent, MODELS_DIR, CHECKPOINTS_DIR, LATENTS_DIR, LOGS_DIR,
        RESULTS_PLOTS_DIR, RESULTS_METRICS_DIR, RESULTS_ECOTRACKER_DIR,
        SYNTHETIC_NEW_POS_DIR, SYNTHETIC_NEW_NEG_DIR,
    ]:
        directory.mkdir(parents=True, exist_ok=True)

    HELPERS = [
        UTILITY_DIR / "prepare_sdvae_latents_v2.py",
        UTILITY_DIR / "train_ldm_v2.py",
        UTILITY_DIR / "evaluate_ldm_v2.py",
        UTILITY_DIR / "generate_ldm_v2.py",
        UTILITY_DIR / "evaluate_filtered_ldm_v2.py",
        UTILITY_DIR / "sd_vae_utils.py",
    ]
    for helper in HELPERS:
        if not helper.exists():
            raise FileNotFoundError(helper)


    def run_and_stream(cmd, log_path, env=None):
        log_path = Path(log_path)
        log_path.parent.mkdir(parents=True, exist_ok=True)
        print("Comando:")
        print(" ".join(map(str, cmd)))
        print("Log:", log_path)
        with open(log_path, "w", encoding="utf-8") as log_file:
            proc = subprocess.Popen(
                [str(x) for x in cmd],
                cwd=str(PROJECT_ROOT),
                stdout=log_file,
                stderr=subprocess.STDOUT,
                env=env or os.environ.copy(),
                start_new_session=True,
            )
        print("PID:", proc.pid)
        with open(log_path, "r", encoding="utf-8", errors="replace") as log_file:
            while proc.poll() is None:
                line = log_file.readline()
                if line:
                    print(line, end="", flush=True)
                else:
                    time.sleep(0.5)
            for line in log_file:
                print(line, end="", flush=True)
        print("Return code:", proc.returncode)
        if proc.returncode != 0:
            raise RuntimeError(f"Comando fallito, controlla il log: {log_path}")


    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("EXPERIMENT_DIR:", EXPERIMENT_DIR)
    print("RESULTS_DIR:", RESULTS_DIR)
    print("PRETRAINED_MODEL_DIR:", PRETRAINED_MODEL_DIR)
    print("Output filtrate:", SYNTHETIC_NEW_DIR)


NameError: name 'phase_should_run' is not defined

In [ ]:
# IDEMPOTENT_PHASE_MODES_V1
TRAIN_MODE = "auto"       # auto | run | skip
GENERATION_MODE = "auto"  # auto | run | skip
EVALUATION_MODE = "auto"  # auto | run | skip | recompute
FILTER_MODE = "auto"      # auto | run | skip | recompute
PLAN_ONLY = False
ALLOW_HEAVY_RETRAIN = False       # must be True for auto mode to retrain from scratch
ALLOW_FULL_REGENERATION = False   # must be True for auto mode to regenerate a full image set

from artifact_phase_planner import plan_experiment, print_plan, phase_should_run
PHASE_MODES = {"training": TRAIN_MODE, "generation": GENERATION_MODE,
               "evaluation": EVALUATION_MODE, "filter": FILTER_MODE}
ALLOW_FLAGS = {"training": ALLOW_HEAVY_RETRAIN, "generation": ALLOW_FULL_REGENERATION}
PHASE_PLAN = plan_experiment(EXPERIMENT_DIR, PHASE_MODES, ALLOW_FLAGS)
print_plan(PHASE_PLAN)


## 4. Dataset preprocessato condiviso


In [ ]:
import pandas as pd

required = [
    DATA_PROCESSED_DIR / "metadata" / "train.csv",
    DATA_PROCESSED_DIR / "metadata" / "val.csv",
    DATA_PROCESSED_DIR / "metadata" / "test.csv",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(f"Dataset preprocessato mancante: {path}")

for split in ["train", "val", "test"]:
    df = pd.read_csv(DATA_PROCESSED_DIR / "metadata" / f"{split}.csv")
    print(split, len(df), df["label"].value_counts().to_dict())


train 2041 {0: 1701, 1: 340}
val 437 {0: 364, 1: 73}
test 438 {0: 365, 1: 73}


## 5. Modello Stable Diffusion 2.1 da Drive


In [ ]:
MODEL_WEIGHT_ALIASES = {
    "text_encoder": ("model.fp16.safetensors", "model.safetensors"),
    "unet": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
    "vae": ("diffusion_pytorch_model.fp16.safetensors", "diffusion_pytorch_model.safetensors"),
}


def download_zip(drive_id, destination, force=False):
    destination = Path(destination)
    if destination.exists() and not force and zipfile.is_zipfile(destination):
        print("Archivio modello già presente:", destination)
        return
    destination.unlink(missing_ok=True)
    gdown.download(id=drive_id, output=str(destination), quiet=False)
    if not destination.exists() or not zipfile.is_zipfile(destination):
        raise RuntimeError(f"Download modello non valido: {destination}")


def find_directory(root, predicate, description):
    root = Path(root)
    for candidate in [root, *(path for path in root.rglob("*") if path.is_dir())]:
        if predicate(candidate):
            return candidate
    raise FileNotFoundError(f"Directory {description} non trovata dentro {root}")


def has_diffusers_structure(model_dir):
    model_dir = Path(model_dir)
    required = ("model_index.json", "scheduler", "tokenizer", "text_encoder", "vae", "unet")
    return all((model_dir / item).exists() for item in required)


def local_sd_model_ready(model_dir):
    model_dir = Path(model_dir)
    weights = (
        model_dir / "text_encoder" / "model.safetensors",
        model_dir / "unet" / "diffusion_pytorch_model.safetensors",
        model_dir / "vae" / "diffusion_pytorch_model.safetensors",
    )
    return has_diffusers_structure(model_dir) and all(path.is_file() for path in weights)


def create_model_weight_copies(model_dir):
    model_dir = Path(model_dir)
    for subfolder, (source_name, target_name) in MODEL_WEIGHT_ALIASES.items():
        source_path = model_dir / subfolder / source_name
        target_path = model_dir / subfolder / target_name
        if target_path.is_symlink():
            target_path.unlink()
        elif target_path.exists():
            continue
        if not source_path.is_file():
            raise FileNotFoundError(f"Peso sorgente non trovato: {source_path}")
        shutil.copy2(source_path, target_path)


def prepare_pretrained_model():
    if local_sd_model_ready(PRETRAINED_MODEL_DIR) and not FORCE_MODEL_REDOWNLOAD:
        print("Modello Stable Diffusion 2.1 già pronto.")
        return PRETRAINED_MODEL_DIR.absolute()

    download_zip(SD21_MODEL_DRIVE_ID, PRETRAINED_MODEL_ZIP_PATH, FORCE_MODEL_REDOWNLOAD)
    with TemporaryDirectory(prefix="mammo_sd21_extract_") as tmp:
        with zipfile.ZipFile(PRETRAINED_MODEL_ZIP_PATH) as archive:
            archive.extractall(tmp)
        source_dir = find_directory(tmp, has_diffusers_structure, "modello Diffusers")
        if PRETRAINED_MODEL_DIR.is_symlink() or PRETRAINED_MODEL_DIR.is_file():
            PRETRAINED_MODEL_DIR.unlink()
        elif PRETRAINED_MODEL_DIR.exists():
            shutil.rmtree(PRETRAINED_MODEL_DIR)
        shutil.copytree(source_dir, PRETRAINED_MODEL_DIR)
        create_model_weight_copies(PRETRAINED_MODEL_DIR)

    if not local_sd_model_ready(PRETRAINED_MODEL_DIR):
        raise FileNotFoundError("Modello Stable Diffusion 2.1 incompleto dopo l'estrazione.")
    return PRETRAINED_MODEL_DIR.absolute()


LOCAL_MODEL_DIR = prepare_pretrained_model()
SD_VAE_MODEL_NAME_OR_PATH = LOCAL_MODEL_DIR
print("Modello locale:", LOCAL_MODEL_DIR)


Modello Stable Diffusion 2.1 già pronto.
Modello locale: /home/fede/Documenti/MammoDiffusion/experiments/20260703_ldm_sdvae_extra1361/pretrained_model/stable-diffusion-2-1-base


## 5. Import e verifica VAE Stable Diffusion


In [ ]:
import torch
from diffusers import AutoencoderKL

sys.path.insert(0, str(NOTEBOOKS_DIR))
from sd_vae_utils import load_sd_vae, resolve_sd_vae_model

SD_VAE_MODEL = resolve_sd_vae_model(PROJECT_ROOT, SD_VAE_MODEL_NAME_OR_PATH)
print("SD_VAE_MODEL:", SD_VAE_MODEL)

# Verifica leggera: carica il solo VAE, stampa scaling factor, poi libera memoria.
vae, vae_device, vae_dtype, scaling_factor = load_sd_vae(SD_VAE_MODEL)
print("VAE device:", vae_device)
print("VAE dtype:", vae_dtype)
print("VAE scaling_factor:", scaling_factor)
del vae
if torch.cuda.is_available():
    torch.cuda.empty_cache()

bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/site-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/site-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/ctypes/__init__.py", line 454, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/home/fede/miniforge3/envs/tf-gpu/lib/python3.11/ctypes/__init__.py", line 376, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No such file or directory


SD_VAE_MODEL: /home/fede/Documenti/MammoDiffusion/experiments/20260703_ldm_sdvae_extra1361/pretrained_model/stable-diffusion-2-1-base
VAE device: cuda
VAE dtype: torch.float16
VAE scaling_factor: 0.18215


## 6. Preparazione latenti SD-VAE


In [ ]:
SDVAE_BATCH_SIZE = 4
FORCE_LATENTS_RECOMPUTE = False

prepare_cmd = [
    sys.executable,
    str(UTILITY_DIR / "prepare_sdvae_latents_v2.py"),
    "--project-root", str(PROJECT_ROOT),
    "--experiment-dir", str(EXPERIMENT_DIR),
    "--batch-size", str(SDVAE_BATCH_SIZE),
    "--results-stage-name", RESULTS_STAGE_NAME,
]
prepare_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
if FORCE_LATENTS_RECOMPUTE:
    prepare_cmd.append("--force-recompute")

run_and_stream(prepare_cmd, LOGS_DIR / "prepare_sdvae_latents.log")


## 7. Training completo U-Net LDM su latenti SD


In [ ]:
# IDEMPOTENT_GUARD_V1:training
if phase_should_run(PHASE_PLAN, "training", PLAN_ONLY):
    TOTAL_STEPS = 150_000
    CHECKPOINT_EVERY = 5_000
    LOG_EVERY = 20
    RESUME_FROM_LATEST = True

    train_cmd = [
        sys.executable,
        str(UTILITY_DIR / "train_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--total-steps", str(TOTAL_STEPS),
        "--checkpoint-every", str(CHECKPOINT_EVERY),
        "--log-every", str(LOG_EVERY),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--skip-latent-encoding",
    ]
    if RESUME_FROM_LATEST:
        train_cmd.append("--resume-from-latest")

    run_and_stream(train_cmd, LOGS_DIR / "ldm_train_sdvae.log", env=training_subprocess_env())

## 8. Evaluation checkpoint sweep


In [ ]:
# IDEMPOTENT_GUARD_V1:evaluation
if phase_should_run(PHASE_PLAN, "evaluation", PLAN_ONLY):
    EVAL_MIN_STEP = 1_000
    N_GEN_PER_CLASS = 100
    EVAL_SAMPLE_STEPS = 100
    EVAL_GUIDANCE_SCALE = 1.5
    EVAL_INCEPTION_BATCH = 8
    EVAL_DECODE_ON_CPU = False
    EVAL_ECO_TRACK = True

    eval_cmd = [
        sys.executable,
        str(UTILITY_DIR / "evaluate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--mode", "both",
        "--min-step", str(EVAL_MIN_STEP),
        "--n-gen-per-class", str(N_GEN_PER_CLASS),
        "--sample-steps", str(EVAL_SAMPLE_STEPS),
        "--guidance-scale", str(EVAL_GUIDANCE_SCALE),
        "--mini-batch", "1",
        "--inception-batch", str(EVAL_INCEPTION_BATCH),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
    ]
    eval_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
    if EVAL_DECODE_ON_CPU:
        eval_cmd.append("--decode-on-cpu")
    if EVAL_ECO_TRACK:
        eval_cmd.append("--eco-track")

    add_generation_parallel_args(eval_cmd)
    run_and_stream(eval_cmd, LOGS_DIR / "ldm_evaluate_sdvae.log")

## 8b. Smoke test multi-GPU isolato (opzionale)

Cella disattivata di default (`RUN_MULTI_GPU_SMOKE = False`). Se attivata, esegue una generazione RAW minima (`SMOKE_N_RAW` immagini, `SMOKE_SAMPLE_STEPS` step) in una directory dedicata e univoca sotto `EXPERIMENT_DIR/smoke_multi_gpu_dynamic/run_<time_ns>/`, per verificare che la generazione multi-GPU usi realmente almeno 2 worker/GPU prima di lanciare la generazione reale della sezione 9. Non tocca le directory RAW/FILTERED canoniche e non avvia filtro, validazione, test o reverse diffusion.


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
if phase_should_run(PHASE_PLAN, "generation", PLAN_ONLY):
    RUN_MULTI_GPU_SMOKE = False
    SMOKE_GENERATION_GPU_DEVICES = "0,1"
    SMOKE_MAX_WORKERS = 2
    SMOKE_N_RAW = 16
    SMOKE_SAMPLE_STEPS = 2
    SMOKE_GENERATION_SCHEDULER = "dynamic_reservations"
    SMOKE_RESERVATION_SIZE = 4
    SMOKE_GUIDANCE_SCALE = 1.5
    SMOKE_TARGET_LABEL = 1
    SMOKE_PARAMETERIZATION = "eps"
    SMOKE_UNET_VERSION = "v2"
    SMOKE_VAE_SOURCE = "sd_vae_original"

    if not RUN_MULTI_GPU_SMOKE:
        print("RUN_MULTI_GPU_SMOKE=False: smoke multi-GPU non eseguito (default).")
    else:
        if str(UTILITY_DIR) not in sys.path:
            sys.path.insert(0, str(UTILITY_DIR))
        from generate_ldm_v2 import is_readable_png, resolve_model_path
        from parallel_generation_utils import print_gpu_resolution_dry_run

        print("CUDA_VISIBLE_DEVICES ereditato:", os.environ.get("CUDA_VISIBLE_DEVICES"))
        print("GENERATION_GPU_DEVICES richiesto (smoke):", SMOKE_GENERATION_GPU_DEVICES)
        # Lista esplicita, non "auto": lo smoke fissa le GPU indipendentemente da
        # GENERATION_GPU_DEVICES/CUDA_VISIBLE_DEVICES usati dalla generazione reale.
        smoke_devices = print_gpu_resolution_dry_run(SMOKE_GENERATION_GPU_DEVICES, SMOKE_MAX_WORKERS)
        if len(smoke_devices) < 2:
            raise RuntimeError(
                f"SMOKE MULTI-GPU FALLITO: risolte {len(smoke_devices)} GPU {smoke_devices}, "
                "ne servono almeno 2. Non continuo silenziosamente con una sola GPU."
            )

        SMOKE_MODEL_PATH = resolve_model_path(EXPERIMENT_DIR)
        SMOKE_RUN_DIR = EXPERIMENT_DIR / "smoke_multi_gpu_dynamic" / f"run_{time.time_ns()}"
        SMOKE_RAW_DIR = SMOKE_RUN_DIR / "raw"
        SMOKE_FILTERED_DIR = SMOKE_RUN_DIR / "filtered"
        SMOKE_LOGS_DIR = SMOKE_RUN_DIR / "logs"
        SMOKE_LOGS_DIR.mkdir(parents=True, exist_ok=True)

        # I worker multi-GPU della generazione RAW scrivono sempre sotto
        # EXPERIMENT_DIR/logs/parallel_generation/run_<ts> (create_parallel_run_dir),
        # non sotto SMOKE_RUN_DIR: la troviamo confrontando lo stato prima/dopo.
        parallel_logs_root = Path(EXPERIMENT_DIR).resolve() / "logs" / "parallel_generation"
        runs_before = set(parallel_logs_root.glob("run_*")) if parallel_logs_root.is_dir() else set()

        smoke_cmd = [
            sys.executable,
            str(UTILITY_DIR / "generate_ldm_v2.py"),
            "--project-root", str(PROJECT_ROOT),
            "--experiment-dir", str(EXPERIMENT_DIR),
            "--model-path", str(SMOKE_MODEL_PATH),
            "--mode", "generate",
            "--n-raw", str(SMOKE_N_RAW),
            "--target-label", str(SMOKE_TARGET_LABEL),
            "--raw-dir", str(SMOKE_RAW_DIR),
            "--filtered-dir", str(SMOKE_FILTERED_DIR),
            "--sample-steps", str(SMOKE_SAMPLE_STEPS),
            "--guidance-scale", str(SMOKE_GUIDANCE_SCALE),
            "--vae-backend", "sd",
            "--sd-vae-model", str(SD_VAE_MODEL),
            "--sd-vae-batch-size", "1",
            "--parameterization", SMOKE_PARAMETERIZATION,
            "--unet-version", SMOKE_UNET_VERSION,
            "--vae-source", SMOKE_VAE_SOURCE,
            "--generation-gpus", SMOKE_GENERATION_GPU_DEVICES,
            "--generation-scheduler", SMOKE_GENERATION_SCHEDULER,
            "--generation-reservation-size", str(SMOKE_RESERVATION_SIZE),
            "--max-generation-workers", str(SMOKE_MAX_WORKERS),
        ]
        run_and_stream(smoke_cmd, SMOKE_LOGS_DIR / "smoke_multi_gpu_generate.log")

        expected_smoke_pngs = [SMOKE_RAW_DIR / f"synth_{index:05d}.png" for index in range(SMOKE_N_RAW)]
        actual_smoke_pngs = sorted(SMOKE_RAW_DIR.glob("*.png"))
        if [p.name for p in actual_smoke_pngs] != [p.name for p in expected_smoke_pngs]:
            raise RuntimeError(
                f"SMOKE MULTI-GPU FALLITO: attesi {[p.name for p in expected_smoke_pngs]}, "
                f"trovati {[p.name for p in actual_smoke_pngs]} in {SMOKE_RAW_DIR}"
            )
        unreadable_smoke_pngs = [p.name for p in expected_smoke_pngs if not is_readable_png(p)]
        if unreadable_smoke_pngs:
            raise RuntimeError(f"SMOKE MULTI-GPU FALLITO: PNG non leggibili: {unreadable_smoke_pngs}")

        runs_after = set(parallel_logs_root.glob("run_*"))
        new_runs = sorted(runs_after - runs_before)
        if len(new_runs) != 1:
            raise RuntimeError(
                "SMOKE MULTI-GPU FALLITO: attesa esattamente 1 nuova directory log di run, "
                f"trovate {len(new_runs)}: {new_runs}"
            )
        smoke_run_log_dir = new_runs[0]
        smoke_worker_logs = sorted(smoke_run_log_dir.glob("*_gpu_*.log"))
        if len(smoke_worker_logs) < 2:
            raise RuntimeError(
                f"SMOKE MULTI-GPU FALLITO: attesi almeno 2 log worker distinti in {smoke_run_log_dir}, "
                f"trovati {len(smoke_worker_logs)}: {[p.name for p in smoke_worker_logs]}. "
                "Meno di due GPU/worker sono state effettivamente usate."
            )
        for worker_log in smoke_worker_logs:
            shutil.copy2(worker_log, SMOKE_LOGS_DIR / worker_log.name)

        print("Directory smoke:", SMOKE_RUN_DIR)
        print("Directory log worker:", smoke_run_log_dir)
        print("Log worker trovati:", [p.name for p in smoke_worker_logs])
        print("SMOKE MULTI-GPU SUPERATO")


## 9. Generazione, filtro e test - classe positiva


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
if phase_should_run(PHASE_PLAN, "generation", PLAN_ONLY):
    GEN_N_RAW = 4083
    GEN_N_SELECTED = 1361
    GEN_SAMPLE_STEPS = 100
    GEN_GUIDANCE_SCALE = 1.5
    GEN_MODEL_PATH = CHECKPOINTS_DIR / "ldm_unet_best_eval.keras"
    GEN_DECODE_ON_CPU = False
    GEN_ECO_TRACK = True

    pos_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--mode", "all",
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", "1",
        "--batch-size", "1",
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
    ]
    pos_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
    if GEN_DECODE_ON_CPU:
        pos_cmd.append("--decode-on-cpu")
    if GEN_ECO_TRACK:
        pos_cmd.append("--eco-track")

    add_generation_parallel_args(pos_cmd)
    run_and_stream(pos_cmd, LOGS_DIR / "ldm_generate_positive_sdvae.log")

## 10. Generazione, filtro, validazione e test - classe negativa


In [ ]:
# IDEMPOTENT_GUARD_V1:generation
if phase_should_run(PHASE_PLAN, "generation", PLAN_ONLY):
    NEG_RAW_DIR = EXPERIMENT_DIR / "synthetic_raw_negative"
    NEG_FILTERED_DIR = SYNTHETIC_NEW_NEG_DIR

    neg_base_cmd = [
        sys.executable,
        str(UTILITY_DIR / "generate_ldm_v2.py"),
        "--project-root", str(PROJECT_ROOT),
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--model-path", str(GEN_MODEL_PATH),
        "--n-raw", str(GEN_N_RAW),
        "--n-selected", str(GEN_N_SELECTED),
        "--target-label", "0",
        "--raw-dir", str(NEG_RAW_DIR),
        "--filtered-dir", str(NEG_FILTERED_DIR),
        "--batch-size", "1",
        "--sample-steps", str(GEN_SAMPLE_STEPS),
        "--guidance-scale", str(GEN_GUIDANCE_SCALE),
        "--results-stage-name", RESULTS_STAGE_NAME,
        "--vae-backend", "sd",
    ]
    neg_base_cmd.extend(["--sd-vae-model", str(SD_VAE_MODEL)])
    if GEN_DECODE_ON_CPU:
        neg_base_cmd.append("--decode-on-cpu")
    if GEN_ECO_TRACK:
        neg_base_cmd.append("--eco-track")

    add_generation_parallel_args(neg_base_cmd)
    neg_cmd = [*neg_base_cmd, "--mode", "all"]
    run_and_stream(neg_cmd, LOGS_DIR / "ldm_negative_sdvae_all.log")

## 11. Riepilogo artefatti


In [ ]:
from pathlib import Path
import pandas as pd

summary = {
    "best_eval": CHECKPOINTS_DIR / "ldm_unet_best_eval.keras",
    "latent_stats": LATENTS_DIR / "latent_stats.npz",
    "positive_final_json": RESULTS_METRICS_DIR / "positive" / "final_filtered_vs_test.json",
    "negative_final_json": RESULTS_METRICS_DIR / "negative" / "final_filtered_vs_test.json",
    "positive_filtered_dir": SYNTHETIC_NEW_POS_DIR,
    "negative_filtered_dir": SYNTHETIC_NEW_NEG_DIR,
}
for name, path in summary.items():
    print(f"{name:22s}", path, "OK" if Path(path).exists() else "MISSING")

rows = []
for label_name, directory in [("positive", SYNTHETIC_NEW_POS_DIR), ("negative", SYNTHETIC_NEW_NEG_DIR)]:
    rows.append({
        "class": label_name,
        "directory": str(directory),
        "n_png": len(sorted(Path(directory).glob("*.png"))) if Path(directory).is_dir() else 0,
    })
pd.DataFrame(rows)


best_eval              /home/fede/Documenti/MammoDiffusion/experiments/20260703_ldm_sdvae_extra1361/checkpoints_ldm/ldm_unet_best_eval.keras OK
latent_stats           /home/fede/Documenti/MammoDiffusion/experiments/20260703_ldm_sdvae_extra1361/latents/latent_stats.npz OK
positive_final_json    /home/fede/Documenti/MammoDiffusion/results/04b2_ldm_sdvae_extra1361/metrics/final_filtered_vs_test.json OK
negative_final_json    /home/fede/Documenti/MammoDiffusion/experiments/20260703_ldm_sdvae_extra1361/evaluation_negative/final_filtered_vs_test.json OK
positive_filtered_dir  /home/fede/Documenti/MammoDiffusion/data/synthetic/fromscratch_new/positive OK
negative_filtered_dir  /home/fede/Documenti/MammoDiffusion/data/synthetic/fromscratch_new/negative OK


,class,directory,n_png
0,positive,/home/fede/Documenti/MammoDiffusion/data/synth...,1361
1,negative,/home/fede/Documenti/MammoDiffusion/data/synth...,1361
